In [ ]:
import pandas as pd
from pathlib import Path
%matplotlib inline
%load_ext autoreload
%autoreload 2
from imports import *
import scipy.io
from config import dir_config, ephys_config
from src.utils import ephys_utils
import pickle
from scipy.stats import ttest_rel

compiled_dir = Path(dir_config.data.compiled)

In [ ]:
session_id = "240625_GP_TZ"  # set session_id here

eye_parquet = compiled_dir / session_id / f"{session_id}_eye_tracking_processed.parquet"
timestamp_filename = compiled_dir / session_id / f"{session_id}_timestamps.csv"

In [ ]:
def downsample_and_guassian_smooth(signal, original_rate=30000, target_rate=1000, sigma_ms=5):
    factor = original_rate // target_rate
    n_samples = signal.shape[0] // factor
    # vectorized mean pooling via reshape
    downsampled = np.nanmean(signal[:n_samples * factor].reshape(n_samples, factor), axis=1)
    sigma_samples = int(sigma_ms * target_rate / 1000)
    return scipy.ndimage.gaussian_filter1d(downsampled, sigma=sigma_samples)

## Load from Parquet (use this going forward)

In [ ]:
# eye_parquet = eye_filename.with_suffix(".parquet")

eye_signal = pd.read_parquet(eye_parquet)
timestamps = pd.read_csv(timestamp_filename)

print(eye_signal.shape, eye_signal.dtypes)
print(timestamps.shape, timestamps.dtypes)

In [ ]:
# create a matrix of eye distance for each saccade as a row — fully vectorized
ts = eye_signal.timestamp.values  # assumed sorted and regularly spaced
ex = eye_signal.eye_x.values
ey = eye_signal.eye_y.values

saccades = timestamps.response_onset.dropna().values
window_length = 10501

# find all window start indices at once (one vectorized searchsorted call)
starts = np.searchsorted(ts, saccades - 3000)

# build index matrix via broadcasting: shape (n_saccades, 10501)
idx = starts[:, None] + np.arange(window_length)[None, :]

# mask entries that fall outside the signal
valid = idx < len(ts)
idx_clipped = np.where(valid, idx, 0)

# gather all windows at once
wx = ex[idx_clipped]  # (n_saccades, 10501)
wy = ey[idx_clipped]
wx[~valid] = np.nan
wy[~valid] = np.nan


eye_data = np.sqrt(wx**2 + wy**2)
eye_data[~valid] = np.nan

In [ ]:
eye_data_processed = np.nan * np.ones((eye_data.shape[0], eye_data.shape[1] // 30))
for i in range(eye_data.shape[0]):
    eye_data_processed[i,:] = downsample_and_guassian_smooth(eye_data[i,:], sigma_ms=5)
eye_data_processed = eye_data_processed.T

In [ ]:
eye_data_processed

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
axs[0].plot(eye_data_processed, color="blue", alpha=0.1);
axs[0].set_ylabel("distance (degrees)")
axs[0].axvline(101, color="k", linestyle="--", label="saccade onset")
axs[1].plot(1000*np.diff(eye_data_processed, axis=0), color="red", alpha=0.1);
axs[1].axvline(100, color="k", linestyle="--", label="saccade onset")
axs[1].set_ylabel("velocity (degrees/s)")
axs[2].plot(1e6*np.diff(np.diff(eye_data_processed, axis=0), axis=0), color="green", alpha=0.1);
axs[2].axvline(99, color="k", linestyle="--", label="saccade onset")
axs[2].set_ylabel("acceleration (degrees/s^2)")

In [ ]:

# --- build 0–2 s window aligned to go_onset (30 kHz raw) -----------------
FS_RAW   = 30_000
FS_DOWN  = 1_000
WIN_S    = 2.0
WIN_SAMP = int(WIN_S * FS_RAW)   # 60 000 raw samples

# same trial selection as eye_data_processed (response_onset non-NaN)
go_onsets = timestamps.go_onset.values[~timestamps.response_onset.isna()]
go_starts = np.searchsorted(ts, go_onsets)

idx_go   = go_starts[:, None] + np.arange(WIN_SAMP)[None, :]
valid_go = idx_go < len(ts)
idx_go_c = np.where(valid_go, idx_go, 0)

wx_go = ex[idx_go_c];  wx_go[~valid_go] = np.nan
wy_go = ey[idx_go_c];  wy_go[~valid_go] = np.nan
eye_dist_go = np.sqrt(wx_go**2 + wy_go**2)
eye_dist_go[~valid_go] = np.nan

factor = FS_RAW // FS_DOWN   # 30
n_down = WIN_SAMP // factor  # 2000

def _downsample_smooth(row, factor, sigma_ms=5):
    n = (len(row) // factor) * factor
    d = np.nanmean(row[:n].reshape(-1, factor), axis=1)
    return scipy.ndimage.gaussian_filter1d(d, sigma=sigma_ms)

eye_go_1k = np.vstack([_downsample_smooth(eye_dist_go[i], factor)
                        for i in range(eye_dist_go.shape[0])])  # (n, 2000)

# --- filter on go_onset window --------------------------------------------
good = np.nanmax(eye_go_1k, axis=1) < 5
print(f"{good.sum()} / {len(good)} trials pass (max dist < 5°)")

# --- plot response_onset-aligned window for good trials -------------------
# eye_data_processed is (350, n_trials); 350 samples @ 1 kHz, starts 100 ms before response_onset
n_resp = eye_data_processed.shape[0]          # 350
t_resp = (np.arange(n_resp) - 100)            # –100 … +249 ms

data_good = eye_data_processed[:, good]       # (350, n_good)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(t_resp, data_good, color="steelblue", alpha=0.15, lw=0.6)
ax.plot(t_resp, np.nanmedian(data_good, axis=1), color="k", lw=1.5, label="median")
ax.axvline(0, color="r", lw=1, linestyle="--", label="response onset")
ax.set(xlabel="time from response-onset (ms)", ylabel="eye distance (°)",
       title=f"Good trials (n={good.sum()}), max dist < 5°")
ax.legend()

sort_idx = np.argsort(np.nanmedian(data_good, axis=0))
im = axes[1].imshow(data_good[:, sort_idx].T,
                    aspect="auto", origin="lower", vmin=0, vmax=5,
                    extent=[t_resp[0], t_resp[-1], 0, good.sum()], cmap="viridis")
axes[1].axvline(0, color="r", lw=1, linestyle="--")
axes[1].set(xlabel="time from response-onset (ms)", ylabel="trial (sorted)",
            title="Eye distance heatmap")
plt.colorbar(im, ax=axes[1], label="distance (°)")

plt.tight_layout()
plt.show()
